# IL3.4: Escalabilidad y Sostenibilidad
## Notebook 3: Optimización de Recursos: CPU, Memoria y Red

### Objetivo:
Aprender a optimizar el consumo de recursos (procesamiento asíncrono en CPU y manejo de caché en memoria) para aumentar la capacidad del sistema de agentes y disminuir los costos operacionales de red.

### Optimización de Recursos en Producción:
1. **CPU Optimization:** Implementar procesamiento en paralelo de múltiples requests para no bloquear el hilo de ejecución principal.
2. **Memory Management:** Almacenar en caché las respuestas de cómputo costoso o llamadas al LLM para no repetirlas si la consulta del usuario es idéntica.
3. **Network Optimization:** Reducir el consumo de red mediante connection pooling hacia APIs externas, compresión de payloads y batching.


### Inicialización del Entorno
Ejecuta la siguiente celda para configurar el cliente LLM y el agente real de Wikipedia usando LangChain.


In [4]:
import os
import wikipedia
from langchain_openai import ChatOpenAI

# Configurar el idioma de Wikipedia
wikipedia.set_lang("es")

# Configuración del LLM
try:
    llm = ChatOpenAI(
        model="openai/gpt-oss-120b",
        openai_api_base=os.environ.get("GROQ_BASE_URL"),
        openai_api_key=os.environ.get("GROQ_API_KEY"),
        temperature=0
    )
    print("✅ LLM de LangChain configurado.")
except Exception as e:
    print(f"❌ Error configurando el LLM: {e}")
    llm = None

from langchain_classic.agents import tool, create_openai_tools_agent, AgentExecutor
from langsmith import Client

@tool
def get_wikipedia_summary(query: str) -> str:
    """Busca en Wikipedia un tema y devuelve un resumen de 2 frases. Útil para obtener información sobre personas, lugares o conceptos."""
    try:
        return wikipedia.summary(query, sentences=2)
    except Exception as e:
        return f"Ocurrió un error: {e}"

tools = [get_wikipedia_summary]

client = Client(None)
# Usamos el mismo prompt de la comunidad que ya está preparado para manejar historial
prompt = client.pull_prompt("hwchase17/openai-tools-agent", dangerously_pull_public_prompt=True)

agent = create_openai_tools_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

print("✅ Agente y herramientas listos.")


✅ LLM de LangChain configurado.
✅ Agente y herramientas listos.


### Optimización de CPU con Asyncio y Threads
Utilizaremos un `ThreadPoolExecutor` acoplado a funciones asíncronas para disparar un lote de consultas de forma paralela en el agente de Wikipedia real, evitando el procesamiento bloqueante secuencial.


In [ ]:
import asyncio # maneja las tareas que corren en paralelo
from concurrent.futures import ThreadPoolExecutor # importa el pool de hilos que ejecuta funciones en paralelo
import time # medición de tiempo de ejecución

# Clase OptimizedAgent que maneja la ejecución de consultas en paralelo
class OptimizedAgent:
    def __init__(self, executor):
        self.executor = executor
        # Configurar un pool de hilos para llamadas bloqueantes
        self.thread_pool = ThreadPoolExecutor(max_workers=4)
    # Procesa la consulta de forma individual,
    def process_single(self, req):
        try:
            if llm is None:
                time.sleep(0.3)
                return f"Simulado: {req}"
            res = self.executor.invoke({"input": req})
            return res.get("output", "")
        except Exception as e:
            return f"Error: {e}"

    # Procesa un lote de consultas de forma concurrente usando asyncio
    async def process_batch(self, requests):
        loop = asyncio.get_event_loop()
        # Crear tareas concurrentes en el pool de hilos
        tasks = [
            loop.run_in_executor(self.thread_pool, self.process_single, req)
            for req in requests
        ]
        # Esperar la ejecución de todo el lote de forma no bloqueante
        return await asyncio.gather(*tasks)

# Lote de consultas concurrentes
requests_batch = [
    "¿Quién fue Albert Einstein?",
    "¿Qué es la fotosíntesis?",
    "Busca información sobre las ballenas azules.",
    "Dame datos del Imperio Inca."
]
# crea el objeto OptimizedAgent con el executor del agente
agent_opt = OptimizedAgent(agent_executor)

start = time.time()
print("Procesando lote de consultas concurrentes...")

# Ejecutar lote en el bucle de eventos asíncrono
loop = asyncio.get_event_loop()
results = await agent_opt.process_batch(requests_batch)
end = time.time()

for req, res in zip(requests_batch, results):
    print(f"\nConsulta: '{req}'")
    print(f"Respuesta: {res[:120]}...")

print(f"\nTiempo total de ejecución en paralelo: {end - start:.3f} segundos")

Procesando lote de consultas concurrentes...


> Entering new AgentExecutor chain...


> Entering new AgentExecutor chain...


> Entering new AgentExecutor chain...


> Entering new AgentExecutor chain...

Invoking: `get_wikipedia_summary` with `{'query': 'Blue whale'}`



Invoking: `get_wikipedia_summary` with `{'query': 'Imperio Inca'}`



Invoking: `get_wikipedia_summary` with `{'query': 'Fotosíntesis'}`



Invoking: `get_wikipedia_summary` with `{'query': 'Albert Einstein'}`


Ocurrió un error: Page id "blue whales" does not match any pages. Try another id!
Invoking: `get_wikipedia_summary` with `{'query': 'Ballena azul'}`


La fotosíntesis o función clorofílica es un proceso bioquímico que consiste en la conversión de materia inorgánica a materia orgánica gracias a la energía que aporta la luz solar. En este proceso, la energía lumínica se transforma en energía química, siendo el NADPH (nicotín adenín dinucleótido fosfato) y el ATP (adenosín trifosfato) las primeras moléculas en l

c:\Users\ingri\OneDrive\Documentos\Track\Ingenier-a-de-Soluciones-con-Inteligencia-Artificial\.venv\Lib\site-packages\langchain_core\messages\ai.py:508: RuntimeWarning: coroutine 'OptimizedAgent.process_batch' was never awaited
  @model_validator(mode="after")


**Imperio Inca (Tawantinsuyu)**  

| Aspecto | Detalle |
|---|---|
| **Nombre original** | **Tawantinsuyu** (quechua: *tawa* “cuatro” + *suyu* “región” → “las cuatro regiones”). |
| **Fundador legendario** | **Manco Cápac** (siglo XIII), según la mitología inca. |
| **Primer gran gobernante histórico** | **Pachacútec** (c. 1438‑1471), quien transformó el pequeño reino de Cusco en un imperio. |
| **Periodo de mayor expansión** | **Siglos XV‑XVI** (aprox. 1438‑1533). |
| **Extensión territorial** | ~2 000 000 km². Desde la cuenca del **río Ancasmayo** (actual Colombia) hasta el **río Maule** (Chile). Al oeste limitaba con el Pacífico; al este con la selva amazónica y el Gran Chaco. |
| **División administrativa** | Cuatro *suyus* (regiones): **Chinchaysuyu** (norte‑oeste), **Antisuyu** (norte‑este), **Kuntisuyu** (sur‑oeste) y **Qullasuyu** (sur‑este). Cada *suyu* estaba subdividido en *wamani* (provincias) y a su vez en *ayllus* (comunidades familiares). |
| **Capital** | **Cusco**, con

### Gestión de Memoria mediante Caché LRU (Least Recently Used)
Crearemos una clase envoltoria `MemoryEfficientAgent` que implementa una caché en memoria para respuestas del agente. Si una consulta idéntica ya fue respondida, no volverá a llamar al modelo real ni al scraping de Wikipedia, optimizando memoria, latencia y costos operacionales.


In [7]:
# clase que resive ejecutor del agente, diccionario de cache y tamaño max de caché
class MemoryEfficientAgent:
    def __init__(self, executor, max_cache_size=5):
        self.executor = executor
        self.cache = {}
        self.max_cache_size = max_cache_size
    # resive la consulta y calcula hash(imput_data) para usarlo como key en el diccionario de cache
    def process_with_cache(self, input_data):
        cache_key = hash(input_data)
        
        # Verificar si existe en la memoria caché
        if cache_key in self.cache:
            print(f"[Cache HIT] Entrada recuperada de memoria para: '{input_data}'")
            return self.cache[cache_key]
        
        # Si no existe, invocar al modelo real
        print(f"[Cache MISS] Invocando agente real para: '{input_data}'")
        try:
            if llm is None:
                result = f"Respuesta simulada para {input_data}"
            else:
                response = self.executor.invoke({"input": input_data})
                result = response.get("output", "")
        except Exception as e:
            result = f"Error: {e}"
        
        # Controlar el tamaño del caché con política LRU simple (eliminar primer elemento insertado)
        if len(self.cache) >= self.max_cache_size:
            oldest_key = next(iter(self.cache))
            print(f"[Cache EVICT] Removiendo entrada antigua del caché.")
            del self.cache[oldest_key] # guardar la nueva entrada en el caché
            
        self.cache[cache_key] = result
        return result

# Instanciar agente con caché de tamaño 3
agent_mem = MemoryEfficientAgent(agent_executor, max_cache_size=3)

print("--- Consulta 1 (MISS) ---")
print(agent_mem.process_with_cache("¿Quién fue Galileo Galilei?"))

print("\n--- Consulta 2 (HIT - Repetida) ---")
print(agent_mem.process_with_cache("¿Quién fue Galileo Galilei?"))

print("\n--- Consultas 3 y 4 (MISS - Llenado caché) ---")
agent_mem.process_with_cache("¿Qué es un electrón?")
agent_mem.process_with_cache("Busca datos de los dinosaurios.")

print("\n--- Consulta 5 (MISS - Provocará EVICT de Galileo) ---")
agent_mem.process_with_cache("¿Qué es la Vía Láctea?")

print("\n--- Consulta 6 (MISS de nuevo - Galileo fue eliminado) ---")
print(agent_mem.process_with_cache("¿Quién fue Galileo Galilei?"))


--- Consulta 1 (MISS) ---
[Cache MISS] Invocando agente real para: '¿Quién fue Galileo Galilei?'


> Entering new AgentExecutor chain...

Invoking: `get_wikipedia_summary` with `{'query': 'Galileo Galilei'}`


Galileo di Vincenzo Bonaiuti de' Galilei  (Pisa,  15 de febrero de 1564​-Arcetri, Italia, 8 de enero de 1642)​​ fue un astrónomo, ingeniero,​​ matemático​ y físico italiano, relacionado estrechamente con la revolución científica. Eminente hombre del Renacimiento, mostró interés por casi todas las ciencias y artes (música, literatura, pintura).Galileo Galilei (1564‑1642) fue un astrónomo, ingeniero, matemático y físico italiano, considerado una de las figuras clave de la revolución científica. Sus observaciones telescópicas, como los satélites de Júpiter y las fases de Venus, y su defensa del modelo heliocéntrico cambiaron radicalmente la visión del universo y sentaron las bases de la ciencia moderna.

> Finished chain.
Galileo Galilei (1564‑1642) fue un astrónomo, ingeniero, mate

### Preguntas de Análisis
1. **¿De qué manera la optimización de latencia mediante concurrencia impacta en el costo operativo de procesamiento en lote de prompts?**
2. **¿Qué riesgos e implicaciones de seguridad/privacidad tiene almacenar en caché las respuestas de los usuarios?**
3. **Mencione 3 técnicas adicionales de optimización de red aplicables a la arquitectura de agentes.**
